In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from marine_qc import (
    combine_qc_results,
    do_aground_check,
    do_multiple_sequential_check,
    do_speed_check,
    do_sst_biased_check,
)

In [ ]:
from data import get_buoy_data

# How to use quality control checks with sequential reports from buoys

The ``marine_qc`` toolbox provides some quality control (QC) checks that work on sequential marine buoy reports. Creating some tests dataset we can show how to use them on "real" data. There are three checks that are shown here:

* a speed check that tests that speeds inferred from a sequence of reports are not implausible for a drifting buoy
* a aground check that tests whether reports from a drifting buoy suggest it has run aground and stopped moving
* a biased check for sequences of reports from a drifting buoy that are biased or noisy

Finally, we run all these QC checks within a single function and combine the results of each QC check into a single QC flag.

For more information of all available QC checks on sequential buoy reports see [the overview of QC functions for additional buoy reports](https://marine-qc.readthedocs.io/en/latest/overview_buoy_tracking.html).

The test dataset we provide is a ``pandas.DataFrame`` including sequential marine reports representing the track of one drifting buoy. The QC functions return a QC flag for each individual report. The flags are:

* `0`: the check has passed
* `1`: the check has failed
* `2`: the check is untestable

In [ ]:
data = get_buoy_data()
data

The drifting buoy data include four different parameters (`lat`, `lon`, `date`, and `sst`).

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 8))

axs[0].plot(data["lon"], data["lat"], marker="o", linestyle="-")
axs[0].set_title("Buoy drifter – latitude/longitude track")
axs[0].set_xlabel("Longitude (°)")
axs[0].set_ylabel("Latitude (°)")
axs[0].grid(True)

axs[1].plot(data["sst"], marker="o", linestyle="-")
axs[1].set_title("Buoy drifter – sea surface temperature")
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Temperature (°C)")
axs[1].grid(True)

plt.tight_layout()
plt.show()

We see a buoy that is drifting northwards with a constant speed expect one time step. The sea-surface temperature has an offset in the middle of the track.

Firstly, a speed check is performed. Beside `lat`, `lon`, and `date`, this check needs some extra parameters:

* speed_limit: maximum allowable speed for an in situ drifting buoy in meters per second (`2.5`)
* min_win_period: minimum period of time in days over which position is assessed for speed estimates (`0.5`)
* max_win_period: maximum period of time in days over which position is assessed for speed estimates (`1.0`)

In [ ]:
qc_speed = do_speed_check(
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    speed_limit=2.5,
    min_win_period=0.5,
    max_win_period=1.0,
)
pd.DataFrame({"date": data.date, "lat": data.lat, "lon": data.lon, "qc_speed": qc_speed})

As expected the speed check fails for three time steps where the speed increases.

Next, a aground check is performed. Beside `lat`, `lon`, and `date`, this check needs some extra parameters:

* smooth_win: length of window (odd number) in datapoints used for smoothing lon/lat (`3`)
* min_win_period: minimum period of time in days over which position is assessed for speed estimates (`1`)
* max_win_period: maximum period of time in days over which position is assessed for speed estimates (`2`)

In [ ]:
qc_aground = do_aground_check(
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    smooth_win=3,
    min_win_period=1,
    max_win_period=2,
)
pd.DataFrame({"date": data.date, "lat": data.lat, "lon": data.lon, "qc_aground": qc_aground})

For the last four time steps the buoy is not moving anymore. Hence, the aground check fails as expected.

Now, a bias check is performed on the sea-surface temperature. Beside `lat`, `lon`, `date` and `sst`, this check needs some extra parameters:

* `ostia`: 1-dimensional array of background field sea surface temperatures
* `ice`: 1-dimensional array of ice concentrations in the range 0.0 to 1.0 (`0.0`)
* `bgvar`: 1-dimensional array of background sea surface temperature fields variances in K^^2 (`0.01`)
* `n_eval`: minimum number of drifter observations required to be assessed by the long-record check (`9`)
* `bias_lim`: maximum allowable drifter-background bias, beyond which a record is considered biased in Kelvin (`0.5`)
* `drif_intra`: maximum random measurement uncertainty reasonably expected in drifter data in Kelvin (`1.0`)
* `drif_inter`: spread of biases expected in drifter data in Kelvin (`0.29`)
* `err_std_n`: number of standard deviations of combined background and drifter error, beyond which short-record data are deemed suspicious (`3`)
* `n_bad`: minimum number of suspicious data points required for failure of short-record check (`2`)
* `background_err_lim`: Background error variance beyond which the SST background is deemed unreliable in K^^2 (`0.3`)

In [ ]:
ostia = pd.Series([22.0, 21.6, 21.2, 20.8, 20.4, 20.0, 19.6, 19.2, 18.8, 18.4, 18.0, 17.6, 17.2, 16.8, 16.4, 16.0, 16.0, 16.0, 16.0])
qc_biased = do_sst_biased_check(
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    sst=data.sst,
    ostia=ostia,
    ice=[0.0] * 19,
    bgvar=[0.01] * 19,
    n_eval=9,
    bias_lim=0.5,
    drif_intra=1.0,
    drif_inter=0.29,
    err_std_n=3.0,
    n_bad=2,
    background_err_lim=0.3,
)
pd.DataFrame({"date": data.date, "lat": data.lat, "lon": data.lon, "sst": data.sst, "qc_biased": qc_biased})

If the biased check fails for one observation the QC flags for the entire reports are set to `2` (failed).

Now, we run the two QC checks (speed and aground) within a single function `do_multiple_sequential_check`. Therefore, we need a `pandas.DataFrame` and a nested QC dictionary. This is the structure of the dictionary:

* arbitrary user-specified name for the checks
  * "func": name of the QC function as `str` (mandatory)
  * "names": dictionary containing parameter names of the QC function and their corresponding columns in the `pandas.DataFrame` (mandatory)
  * "arguments": dictionary containing any key-word arguments passed to the QC function (optionally)
* ...

We define the QC dictionary according to the QC checks above.

In [ ]:
qc_dict = {
    "speed_check": {
        "func": "do_speed_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
            "date": "date",
        },
        "arguments": {
            "speed_limit": 2.5,
            "min_win_period": 1,
            "max_win_period": 2,
        },
    },
    "aground_check": {
        "func": "do_aground_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
            "date": "date",
        },
        "arguments": {
            "smooth_win": 3,
            "min_win_period": 1,
            "max_win_period": 2,
        },
    },
}

We set `return_method` to "failed" which means that all requested QC check are run until the first check fails. The other QC checks are flagged as unstested (`3`). Furthermore, we set `groupby` to "buoy_id" which specifies how the data should be grouped before applying QC functions.

In [ ]:
qc_multi = do_multiple_sequential_check(
    data,
    qc_dict=qc_dict,
    groupby="buoy_id",
    return_method="failed",
)
qc_multi

Now, we combine the results into one final QC flag. The QC flag values are prioritized in this order: [1, 0, 3, 2].

In [ ]:
qc_flag = combine_qc_results(qc_multi)
pd.DataFrame({**data, "qc_flag": qc_flag})